In [ ]:
from google.colab import drive
import pandas as pd
import glob

# 1. Connect to Google Drive
drive.mount('/content/drive')

benign_path = "/content/drive/MyDrive/EncryptedSDP/datasets/training/raw/benign-csvs/*.csv"
malic_path = "/content/drive/MyDrive/EncryptedSDP/datasets/training/raw/malic-csvs/*.csv"

# 2. Load, tag, and merge the benign and malicious datasets
benign_dfs = [pd.read_csv(f) for f in glob.glob(benign_path)]
benign_data = pd.concat(benign_dfs, ignore_index=True)
benign_data['Threat_Label'] = 'Benign'

malic_dfs = [pd.read_csv(f) for f in glob.glob(malic_path)]
malic_data = pd.concat(malic_dfs, ignore_index=True)
malic_data['Threat_Label'] = 'Malicious'

full_train_df = pd.concat([benign_data, malic_data], ignore_index=True)

# 3. Extract the column names
print("--- EXACT NETWORK FEATURES (COLUMNS) ---")
for i, col in enumerate(full_train_df.columns, 1):
    print(f"{i}. {col}")

# 4. Preview Benign Traffic (The Top Rows)
print("\n--- PREVIEW OF BENIGN TRAFFIC (First 3 Rows) ---")
display(full_train_df.head(3))

# 5. Preview Malicious Traffic (The Bottom Rows)
print("\n--- PREVIEW OF MALICIOUS TRAFFIC (Last 3 Rows) ---")
display(full_train_df.tail(3))

In [ ]:
print("Starting dataset cleaning process...\n")

# 1. Define the exact identifier columns to delete
columns_to_drop = ['SourceIP', 'DestinationIP', 'SourcePort', 'DestinationPort', 'TimeStamp']

# 2. Drop these columns to prevent model overfitting
clean_train_df = full_train_df.drop(columns=columns_to_drop)

# 3. Check for and remove any corrupted or missing data (NaN values)
initial_rows = len(clean_train_df)
clean_train_df = clean_train_df.dropna()
dropped_rows = initial_rows - len(clean_train_df)

# 4. Print the final ML readiness report
print("--- FINAL DATASET CLEANING REPORT ---")
print(f"Identifiers Dropped: {columns_to_drop}")
print(f"Corrupted/Blank Rows Removed: {dropped_rows:,}")
print(f"Total ML-Ready Rows: {len(clean_train_df):,}")
print(f"Total ML-Ready Features: {len(clean_train_df.columns)}")

In [ ]:
# 1. Convert text labels to binary numbers (0 = Safe, 1 = Attack)
clean_train_df['Threat_Label'] = clean_train_df['Threat_Label'].map({'Benign': 0, 'Malicious': 1})

# 2. Save this master dataset permanently to your Drive
save_path = "/content/drive/MyDrive/EncryptedSDP/datasets/training/clean_training_data.csv"
print("Saving clean master dataset to Google Drive... (This takes 1-2 minutes)\n")

clean_train_df.to_csv(save_path, index=False)

print("SUCCESS! Master dataset saved to:", save_path)
print("\n--- FINAL LABEL VERIFICATION ---")
print(clean_train_df['Threat_Label'].value_counts())

In [ ]:
from sklearn.model_selection import train_test_split

print("Configuring data for Machine Learning...\n")

# 1. Separate Features (X) from Target Answers (y)
X = clean_train_df.drop(columns=['Threat_Label'])
y = clean_train_df['Threat_Label']

# 2. Perform the 80/20 split
# stratify=y ensures the 80/20 split maintains the exact same ratio of benign/malicious traffic
X_train, X_val, y_train, y_val = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

print("--- 80/20 DATA SPLIT COMPLETE ---")
print(f"Training Data (X_train) - 80%:   {len(X_train):,} rows")
print(f"Validation Data (X_val) - 20%:   {len(X_val):,} rows")
print(f"Features per row: {X_train.shape[1]}")

In [ ]:

import pandas as pd

print("--- CHECKING BCCC FEATURE ALIGNMENT ---")

bccc_path = "/content/drive/MyDrive/EncryptedSDP/datasets/testing/raw/BCCC-CIRA-CIC-DoHBrw-2020.csv"

bccc_preview = pd.read_csv(bccc_path, nrows=0)

bccc_features = bccc_preview.drop(
    columns=["Label"], errors="ignore"
).columns.tolist()

# Remove the two features not available in BCCC
X_train_28 = X_train.drop(
    columns=["Duration", "DoH"], errors="ignore"
)

X_val_28 = X_val.drop(
    columns=["Duration", "DoH"], errors="ignore"
)

missing_features = set(bccc_features) - set(X_train_28.columns)
extra_features = set(X_train_28.columns) - set(bccc_features)

assert not missing_features, f"Missing features: {missing_features}"
assert not extra_features, f"Extra features: {extra_features}"

# Ensure the same feature order
X_train_28 = X_train_28[bccc_features]
X_val_28 = X_val_28[bccc_features]

print("Feature alignment successful!")
print("Training records:", len(X_train_28))
print("Validation records:", len(X_val_28))
print("Total features:", X_train_28.shape[1])


In [ ]:

import time
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.ensemble import RandomForestClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    classification_report,
    confusion_matrix
)

models = {
    "Random Forest": RandomForestClassifier(
        n_estimators=100,
        random_state=42,
        n_jobs=-1
    ),

    "Decision Tree": DecisionTreeClassifier(
        random_state=42
    ),

    "Logistic Regression": make_pipeline(
        StandardScaler(),
        LogisticRegression(
            max_iter=1000,
            random_state=42
        )
    )
}

results = []
trained_models = {}

for name, model in models.items():

    print("\n=================================")
    print("TRAINING:", name)
    print("=================================")

    start_time = time.time()

    model.fit(X_train_28, y_train)

    training_time = time.time() - start_time

    y_pred = model.predict(X_val_28)

    accuracy = accuracy_score(y_val, y_pred)
    precision = precision_score(y_val, y_pred)
    recall = recall_score(y_val, y_pred)
    f1 = f1_score(y_val, y_pred)

    trained_models[name] = model

    results.append({
        "Algorithm": name,
        "Accuracy (%)": round(accuracy * 100, 4),
        "Precision": round(precision, 6),
        "Recall": round(recall, 6),
        "F1-Score": round(f1, 6),
        "Training Time (s)": round(training_time, 2)
    })

    print("\nTraining Time:", round(training_time, 2), "seconds")
    print("\nClassification Report:")
    print(classification_report(
        y_val,
        y_pred,
        target_names=["Benign", "Malicious"]
    ))

    cm = confusion_matrix(y_val, y_pred)

    plt.figure(figsize=(6, 4))
    sns.heatmap(
        cm,
        annot=True,
        fmt="d",
        cmap="Blues",
        xticklabels=["Benign", "Malicious"],
        yticklabels=["Benign", "Malicious"]
    )

    plt.title(name + " - 80:20 Validation")
    plt.xlabel("Predicted Label")
    plt.ylabel("Actual Label")
    plt.show()

print("\nFINAL ALGORITHM COMPARISON")

comparison_df = pd.DataFrame(results)

comparison_df = comparison_df.sort_values(
    by=["F1-Score", "Recall", "Accuracy (%)"],
    ascending=False
)

display(comparison_df)

best_name = comparison_df.iloc[0]["Algorithm"]
best_model = trained_models[best_name]

print("\nBEST PERFORMING ALGORITHM:", best_name)


In [ ]:

import joblib
import os

print("Best Algorithm:", best_name)

if best_name == "Random Forest":

    rf_model_aligned = trained_models["Random Forest"]

    model_save_path = (
        "/content/drive/MyDrive/EncryptedSDP/notebooks/"
        "random_forest_aligned_28f.pkl"
    )

    os.makedirs(
        os.path.dirname(model_save_path),
        exist_ok=True
    )

    joblib.dump(rf_model_aligned, model_save_path)

    print("\nFINAL MODEL SAVED SUCCESSFULLY!")
    print("Model:", best_name)
    print("Features:", rf_model_aligned.n_features_in_)
    print("Saved at:", model_save_path)

else:
    print("\nRandom Forest was not the top-ranked model.")
    print("Review the comparison before choosing a deployment model.")


In [ ]:

from sklearn.metrics import (
    confusion_matrix,
    classification_report,
    accuracy_score
)
import matplotlib.pyplot as plt
import seaborn as sns

# Use the selected Random Forest model
rf_model = trained_models["Random Forest"]

# Predict on the 20% validation dataset
y_pred_rf = rf_model.predict(X_val_28)

# Calculate accuracy
accuracy = accuracy_score(y_val, y_pred_rf)

print("RANDOM FOREST - 80:20 VALIDATION RESULTS")
print("=" * 45)

print(f"Training Records: {len(X_train_28):,}")
print(f"Validation Records: {len(X_val_28):,}")
print(f"Total Features: {X_val_28.shape[1]}")
print(f"Accuracy: {accuracy * 100:.4f}%")

print("\nCLASSIFICATION REPORT")
print(classification_report(
    y_val,
    y_pred_rf,
    target_names=["Benign", "Malicious"],
    digits=4
))

# Generate confusion matrix
cm = confusion_matrix(y_val, y_pred_rf, labels=[0, 1])

plt.figure(figsize=(8, 6))

sns.heatmap(
    cm,
    annot=True,
    fmt="d",
    cmap="Blues",
    xticklabels=["Benign", "Malicious"],
    yticklabels=["Benign", "Malicious"]
)

plt.title(
    "Random Forest - Confusion Matrix (80:20 Split)",
    fontsize=14,
    fontweight="bold"
)

plt.xlabel("Predicted Label")
plt.ylabel("Actual Label")

plt.tight_layout()

# Save confusion matrix to Google Drive
save_path = (
    "/content/drive/MyDrive/EncryptedSDP/notebooks/"
    "rf_80_20_confusion_matrix.png"
)

plt.savefig(save_path, dpi=300, bbox_inches="tight")
plt.show()

print("\nConfusion matrix saved successfully!")
print("Location:", save_path)
